# rippl — Full Validation Benchmark Suite v2
**All implemented physics systems. T4 GPU required.**
Run all cells top to bottom. Results saved to `benchmark_results.json`.

In [ ]:
# CELL 1 — Setup
!git clone https://github.com/esranow/rippl.git 2>/dev/null || (cd rippl && git pull)
%cd rippl
!pip install -e '.[fractional]' -q
!pip install deepxde -q
import sys
sys.path.insert(0, '/content/rippl')
import torch, math, time, json, warnings
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
results = {}

In [ ]:
# CELL 2 — Import health check
try:
    from rippl.core import System, Domain, Constraint, Equation, EquationSystem, Experiment
    from rippl.core import ReferenceScales, NondimSystem, DigitalTwin
    from rippl.physics.operators import Laplacian, TimeDerivative, BurgersAdvection, Diffusion, ArtificialViscosity
    from rippl.physics.navier_stokes import NavierStokesSystem
    from rippl.physics.schrodinger import SchrodingerSystem
    from rippl.physics.phase_field import AllenCahnOperator, PhaseFieldSystem
    from rippl.physics.reaction_diffusion import TuringSystem
    from rippl.physics.hamilton_jacobi import EikonalOperator, HJSystem
    from rippl.physics.distance import BoxDistance, HardConstraintWrapper, AnsatzFactory
    from rippl.physics.conservative import StreamFunctionModel
    from rippl.training.causal import CausalTrainingMixin
    from rippl.training.ntk_weighting import AdaptiveLossBalancer
    from rippl.training.pinn_recipe import PINNTrainingRecipe
    from rippl.training.uq import MCDropoutWrapper, UncertaintyQuantifier
    from rippl.training.fno_flywheel import FNOFlywheel
    from rippl.geometry.csg import Circle, Annulus, CSGSampler, CSGDomain
    from rippl.data.sensor import SensorDataset, MultiFidelityFusion
    from rippl.models.mlp import MLP
    from rippl.models.siren import Siren
    from rippl.models.fno import FNO1d
    from rippl.physics_blocks.residual import HybridWaveResidualBlock
    print('ALL IMPORTS OK')
except ImportError as e:
    print(f'IMPORT FAILED: {e}')

In [ ]:
# CELL 3 — Shared utilities
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=50, layers=4):
        super().__init__()
        dims = [in_dim] + [hidden]*layers + [out_dim]
        mods = []
        for i in range(len(dims)-1):
            mods.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims)-2: mods.append(nn.Tanh())
        self.net = nn.Sequential(*mods)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    def forward(self, x): return self.net(x)

def rel_l2(pred, true):
    return ((pred-true).pow(2).sum().sqrt() / true.pow(2).sum().sqrt()).item()

def sobol(n, d): 
    s = torch.quasirandom.SobolEngine(d, scramble=True)
    return s.draw(n).to(device)

def adam_train(model, loss_fn, epochs=5000, lr=5e-4, clip=1.0, log_every=1000):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=200, factor=0.5, min_lr=1e-6)
    history = []
    for ep in range(epochs):
        opt.zero_grad()
        loss = loss_fn()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
        sched.step(loss)
        history.append(loss.item())
        if ep % log_every == 0:
            print(f'  ep {ep}: {loss.item():.3e} lr={opt.param_groups[0]["lr"]:.1e}')
    return history

def lbfgs_refine(model, loss_fn, steps=300):
    opt = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=20, line_search_fn='strong_wolfe')
    for s in range(steps):
        def closure():
            opt.zero_grad()
            l = loss_fn()
            l.backward()
            return l
        opt.step(closure)
        if s % 100 == 0:
            with torch.no_grad(): print(f'  LBFGS {s}: {loss_fn().item():.3e}')

print('Utilities ready')

## B1 — Physics Sanity Check

In [ ]:
# CELL 4 — Physics sanity
print('=== B1: Physics Sanity ===')
from rippl.physics_blocks.residual import HybridWaveResidualBlock
from rippl.core.equation import Equation as Eq
from rippl.physics.operators import TimeDerivative, Laplacian
coords = torch.rand(1000, 2, requires_grad=True, device=device)
u = torch.sin(math.pi*coords[:,0:1]) * torch.cos(math.pi*coords[:,1:2])
eq = Eq([TimeDerivative(2), Laplacian(spatial_dims=1)])
block = HybridWaveResidualBlock(a=1.0, b=0.0, c=1.0, use_correction=False)
res = block.residual(u, eq, coords)
val = res.abs().max().item()
status = 'PASS' if val < 1e-2 else 'FAIL'
print(f'Residual: {val:.2e} — {status}')
results['b1_sanity'] = {'residual': val, 'status': status}

## B2 — Heat Equation 1D

In [ ]:
# CELL 5 — Heat 1D: u_t = 0.1*u_xx, analytic: sin(πx)*exp(-0.1π²t)
print('=== B2: Heat 1D ===')
alpha = 0.1
m = MLP(2, 1, 50, 5).to(device)
col = sobol(10000, 2)
x_ic = torch.rand(2000,1,device=device)
c_ic = torch.cat([x_ic, torch.zeros(2000,1,device=device)],1)
u_ic = torch.sin(math.pi*x_ic)
t_bc = torch.rand(2000,1,device=device)
c_bcl = torch.cat([torch.zeros(2000,1,device=device),t_bc],1)
c_bcr = torch.cat([torch.ones(2000,1,device=device),t_bc],1)
u_bc = torch.zeros(2000,1,device=device)

def heat_loss():
    c = col.clone().requires_grad_(True)
    u = m(c)
    du = torch.autograd.grad(u.sum(),c,create_graph=True)[0]
    u_t,u_x = du[:,1:2],du[:,0:1]
    u_xx = torch.autograd.grad(u_x.sum(),c,create_graph=True)[0][:,0:1]
    res = (u_t - alpha*u_xx).pow(2).mean()
    l_ic = F.mse_loss(m(c_ic),u_ic)
    l_bc = F.mse_loss(m(c_bcl),u_bc)+F.mse_loss(m(c_bcr),u_bc)
    return res + 100*(l_ic+l_bc)

t0=time.time()
adam_train(m, heat_loss, 8000, 5e-4)
lbfgs_refine(m, heat_loss, 500)
elapsed=time.time()-t0

xe=torch.linspace(0,1,200,device=device)
te=torch.linspace(0,1,200,device=device)
X,T=torch.meshgrid(xe,te,indexing='ij')
ce=torch.stack([X.flatten(),T.flatten()],1)
with torch.no_grad(): up=m(ce).reshape(200,200).cpu()
ut=(torch.sin(math.pi*X)*torch.exp(-alpha*math.pi**2*T)).cpu()
l2=rel_l2(up,ut)
status='PASS' if l2<1e-2 else 'FAIL'
print(f'L2={l2:.4e} {status} {elapsed:.0f}s')
results['b2_heat1d']={'l2':l2,'time':elapsed,'status':status}

fig,ax=plt.subplots(1,3,figsize=(14,3))
ax[0].contourf(T.cpu(),X.cpu(),ut,50,cmap='RdBu_r'); ax[0].set_title('Analytic')
ax[1].contourf(T.cpu(),X.cpu(),up,50,cmap='RdBu_r'); ax[1].set_title('rippl')
ax[2].contourf(T.cpu(),X.cpu(),(up-ut).abs(),50,cmap='hot_r'); ax[2].set_title(f'Error L2={l2:.2e}')
plt.tight_layout(); plt.savefig('b2_heat1d.png',dpi=100); plt.show()

## B3 — Wave Equation 1D (Causal)

In [ ]:
# CELL 6 — Wave 1D with causal training, analytic: sin(πx)cos(πt)
print('=== B3: Wave 1D Causal ===')
m_w = MLP(2,1,100,6).to(device)
col_w = sobol(10000,2)
x_ic=torch.rand(3000,1,device=device)
c_ic_w=torch.cat([x_ic,torch.zeros(3000,1,device=device)],1)
u_ic_w=torch.sin(math.pi*x_ic)
t_bc_w=torch.rand(3000,1,device=device)
c_bcl_w=torch.cat([torch.zeros(3000,1,device=device),t_bc_w],1)
c_bcr_w=torch.cat([torch.ones(3000,1,device=device),t_bc_w],1)
u_bc_w=torch.zeros(3000,1,device=device)

def causal_wave_loss(epsilon=1.0, n_bins=20):
    c=col_w.clone().requires_grad_(True)
    u=m_w(c)
    du=torch.autograd.grad(u.sum(),c,create_graph=True)[0]
    u_t,u_x=du[:,1:2],du[:,0:1]
    u_tt=torch.autograd.grad(u_t.sum(),c,create_graph=True)[0][:,1:2]
    u_xx=torch.autograd.grad(u_x.sum(),c,create_graph=True)[0][:,0:1]
    res=(u_tt-u_xx).pow(2)
    t_vals=c[:,1].detach()
    bins=torch.linspace(t_vals.min(),t_vals.max(),n_bins+1,device=device)
    bl=[res[(t_vals>=bins[i])&(t_vals<bins[i+1])].mean() if ((t_vals>=bins[i])&(t_vals<bins[i+1])).sum()>0 else torch.tensor(0.,device=device) for i in range(n_bins)]
    cum=0.; cw=[]
    for b in bl: cw.append(math.exp(-epsilon*cum)); cum+=b.detach().item()
    loss_r=sum(w*b for w,b in zip(cw,bl))/n_bins
    c_v=c_ic_w.clone().requires_grad_(True)
    u_v=m_w(c_v)
    u_tv=torch.autograd.grad(u_v.sum(),c_v,create_graph=True)[0][:,1:2]
    l_ic=F.mse_loss(m_w(c_ic_w),u_ic_w)
    l_vel=u_tv.pow(2).mean()
    l_bc=F.mse_loss(m_w(c_bcl_w),u_bc_w)+F.mse_loss(m_w(c_bcr_w),u_bc_w)
    return loss_r+100*(l_ic+l_vel+l_bc)

# Phase A
opt_a=torch.optim.Adam(m_w.parameters(),lr=1e-3)
print('Phase A...')
for ep in range(3000):
    opt_a.zero_grad()
    c_v=c_ic_w.clone().requires_grad_(True)
    u_v=m_w(c_v)
    u_tv=torch.autograd.grad(u_v.sum(),c_v,create_graph=True)[0][:,1:2]
    l=F.mse_loss(m_w(c_ic_w),u_ic_w)+u_tv.pow(2).mean()+F.mse_loss(m_w(c_bcl_w),u_bc_w)+F.mse_loss(m_w(c_bcr_w),u_bc_w)
    l.backward(); torch.nn.utils.clip_grad_norm_(m_w.parameters(),1.0); opt_a.step()
    if ep%1000==0: print(f'  A ep{ep}: {l.item():.3e}')

# Phase B
print('Phase B...')
eps=1.0
t0=time.time()
adam_train(m_w, lambda: causal_wave_loss(eps), 10000, 5e-4)
lbfgs_refine(m_w, lambda: causal_wave_loss(eps), 300)
elapsed_w=time.time()-t0

X_w,T_w=torch.meshgrid(torch.linspace(0,1,200,device=device),torch.linspace(0,1,200,device=device),indexing='ij')
ce_w=torch.stack([X_w.flatten(),T_w.flatten()],1)
with torch.no_grad(): up_w=m_w(ce_w).reshape(200,200).cpu()
ut_w=(torch.sin(math.pi*X_w)*torch.cos(math.pi*T_w)).cpu()
l2_w=rel_l2(up_w,ut_w)
status_w='PASS' if l2_w<1e-2 else 'FAIL'
print(f'L2={l2_w:.4e} {status_w} {elapsed_w:.0f}s')
results['b3_wave1d']={'l2':l2_w,'time':elapsed_w,'status':status_w}

fig,ax=plt.subplots(1,3,figsize=(14,3))
ax[0].contourf(T_w.cpu(),X_w.cpu(),ut_w,50,cmap='RdBu_r'); ax[0].set_title('Analytic')
ax[1].contourf(T_w.cpu(),X_w.cpu(),up_w,50,cmap='RdBu_r'); ax[1].set_title('rippl')
ax[2].contourf(T_w.cpu(),X_w.cpu(),(up_w-ut_w).abs(),50,cmap='hot_r'); ax[2].set_title(f'Error L2={l2_w:.2e}')
plt.tight_layout(); plt.savefig('b3_wave1d.png',dpi=100); plt.show()

## B4 — Burgers Equation 1D

In [ ]:
# CELL 7 — Burgers: u_t + u*u_x = nu*u_xx, nu=0.01/pi
# IC: u(x,0) = -sin(πx), BC: u(-1,t)=u(1,t)=0
print('=== B4: Burgers 1D ===')
nu = 0.01/math.pi
m_b = MLP(2,1,50,5).to(device)
# Sobol on [-1,1]x[0,1]
col_b = sobol(8000,2)*torch.tensor([2.,1.],device=device) - torch.tensor([1.,0.],device=device)
x_ic_b=torch.rand(2000,1,device=device)*2-1
c_ic_b=torch.cat([x_ic_b,torch.zeros(2000,1,device=device)],1)
u_ic_b=-torch.sin(math.pi*x_ic_b)
t_bc_b=torch.rand(2000,1,device=device)
c_bcl_b=torch.cat([-torch.ones(2000,1,device=device),t_bc_b],1)
c_bcr_b=torch.cat([torch.ones(2000,1,device=device),t_bc_b],1)
u_bc_b=torch.zeros(2000,1,device=device)

def burgers_loss():
    c=col_b.clone().requires_grad_(True)
    u=m_b(c)
    du=torch.autograd.grad(u.sum(),c,create_graph=True)[0]
    u_t,u_x=du[:,1:2],du[:,0:1]
    u_xx=torch.autograd.grad(u_x.sum(),c,create_graph=True)[0][:,0:1]
    res=(u_t+u*u_x-nu*u_xx).pow(2).mean()
    l_ic=F.mse_loss(m_b(c_ic_b),u_ic_b)
    l_bc=F.mse_loss(m_b(c_bcl_b),u_bc_b)+F.mse_loss(m_b(c_bcr_b),u_bc_b)
    return res+100*(l_ic+l_bc)

t0=time.time()
adam_train(m_b, burgers_loss, 10000, 5e-4)
lbfgs_refine(m_b, burgers_loss, 500)
elapsed_b=time.time()-t0

# Reference: use scipy for Burgers reference solution
from scipy.integrate import solve_ivp
Nx=256; x_ref=np.linspace(-1,1,Nx); dx=x_ref[1]-x_ref[0]
u0_ref=-np.sin(np.pi*x_ref)
def burgers_rhs(t,u):
    du=np.gradient(u,dx); d2u=np.gradient(du,dx)
    return -u*du+nu*d2u
sol=solve_ivp(burgers_rhs,[0,0.75],u0_ref,t_eval=[0.25,0.5,0.75],max_step=1e-3)

t_eval=[0.25,0.5,0.75]; errors=[]
for i,t_val in enumerate(t_eval):
    x_e=torch.tensor(x_ref,dtype=torch.float32,device=device).unsqueeze(1)
    t_e=torch.full_like(x_e,t_val)
    ce=torch.cat([x_e,t_e],1)
    with torch.no_grad(): up=m_b(ce).cpu().numpy().flatten()
    ref=sol.y[:,i]
    err=np.sqrt(((up-ref)**2).mean())/np.sqrt((ref**2).mean())
    errors.append(err)
    print(f'  t={t_val}: L2={err:.4e}')
mean_l2=np.mean(errors)
status_b='PASS' if mean_l2<5e-2 else 'FAIL'
print(f'Mean L2={mean_l2:.4e} {status_b} {elapsed_b:.0f}s')
results['b4_burgers1d']={'l2_t025':errors[0],'l2_t050':errors[1],'l2_t075':errors[2],'mean_l2':mean_l2,'time':elapsed_b,'status':status_b}


## B5 — Allen-Cahn 1D

In [ ]:
# CELL 8 — Allen-Cahn: phi_t = -M*(f'(phi) - eps²*phi_xx)
# f'(phi) = phi*(1-phi)*(1-2phi), eps=0.1, M=1
# IC: phi(x,0) = 0.5*(1+tanh(x/eps))
print('=== B5: Allen-Cahn 1D ===')
eps_ac=0.1; M_ac=1.0
m_ac=MLP(2,1,64,5).to(device)
# Domain x in [-1,1], t in [0,1], Sobol scaled
col_ac=sobol(8000,2)*torch.tensor([2.,1.],device=device)-torch.tensor([1.,0.],device=device)
x_ic_ac=torch.rand(2000,1,device=device)*2-1
c_ic_ac=torch.cat([x_ic_ac,torch.zeros(2000,1,device=device)],1)
u_ic_ac=0.5*(1+torch.tanh(x_ic_ac/eps_ac))

def ac_loss():
    c=col_ac.clone().requires_grad_(True)
    phi=m_ac(c)
    dphi=torch.autograd.grad(phi.sum(),c,create_graph=True)[0]
    phi_t=dphi[:,1:2]
    phi_x=dphi[:,0:1]
    phi_xx=torch.autograd.grad(phi_x.sum(),c,create_graph=True)[0][:,0:1]
    f_prime=phi*(1-phi)*(1-2*phi)
    res=(phi_t+M_ac*(f_prime-eps_ac**2*phi_xx)).pow(2).mean()
    l_ic=F.mse_loss(m_ac(c_ic_ac),u_ic_ac)
    return res+100*l_ic

t0=time.time()
adam_train(m_ac, ac_loss, 8000, 5e-4)
lbfgs_refine(m_ac, ac_loss, 300)
elapsed_ac=time.time()-t0

# Verify phase separation: phi should be close to 0 or 1 at t=1
x_eval=torch.linspace(-1,1,500,device=device).unsqueeze(1)
t_eval_ac=torch.ones(500,1,device=device)
ce_ac=torch.cat([x_eval,t_eval_ac],1)
with torch.no_grad(): phi_final=m_ac(ce_ac).cpu().numpy().flatten()
# Check: values should be mostly near 0 or 1
near_boundary=np.mean((phi_final<0.1)|(phi_final>0.9))
# Check IC satisfaction
with torch.no_grad(): phi_ic_pred=m_ac(c_ic_ac).cpu()
l2_ic=rel_l2(phi_ic_pred, u_ic_ac.cpu())
status_ac='PASS' if l2_ic<5e-2 and near_boundary>0.5 else 'FAIL'
print(f'IC L2={l2_ic:.4e} phase_sep={near_boundary:.2f} {status_ac} {elapsed_ac:.0f}s')
results['b5_allen_cahn']={'ic_l2':l2_ic,'phase_separation_frac':float(near_boundary),'time':elapsed_ac,'status':status_ac}

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
with torch.no_grad(): phi_ic_plot=m_ac(c_ic_ac).cpu().numpy().flatten()
plt.scatter(x_ic_ac.cpu(),u_ic_ac.cpu(),s=1,label='True IC')
plt.scatter(x_ic_ac.cpu(),phi_ic_plot,s=1,label='rippl IC')
plt.title('IC fit'); plt.legend()
plt.subplot(1,2,2)
plt.plot(x_eval.cpu(),phi_final); plt.title('phi at t=1 (should show phase separation)')
plt.axhline(0,c='k',ls='--'); plt.axhline(1,c='k',ls='--')
plt.tight_layout(); plt.savefig('b5_allen_cahn.png',dpi=100); plt.show()

## B6 — Schrödinger Equation (Particle in Box)

In [ ]:
# CELL 9 — Schrödinger: iψ_t = -(1/2)ψ_xx + V*ψ, V=0 in [0,1]
# Split: psi_real, psi_imag. Ground state E1=π²/2
# Analytic: psi_r=sin(πx)cos(E1*t), psi_i=-sin(πx)sin(E1*t)
print('=== B6: Schrödinger Particle in Box ===')
E1=math.pi**2/2

class TwoHeadMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk=nn.Sequential(nn.Linear(2,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh())
        self.hr=nn.Linear(64,1); self.hi=nn.Linear(64,1)
        for m in self.modules():
            if isinstance(m,nn.Linear): nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self,x):
        h=self.trunk(x); return self.hr(h),self.hi(h)

m_sc=TwoHeadMLP().to(device)
col_sc=sobol(8000,2)
x_ic_sc=torch.rand(2000,1,device=device)
c_ic_sc=torch.cat([x_ic_sc,torch.zeros(2000,1,device=device)],1)
pr_ic=torch.sin(math.pi*x_ic_sc)
pi_ic=torch.zeros_like(pr_ic)
t_bc_sc=torch.rand(2000,1,device=device)
c_bcl_sc=torch.cat([torch.zeros(2000,1,device=device),t_bc_sc],1)
c_bcr_sc=torch.cat([torch.ones(2000,1,device=device),t_bc_sc],1)
u_bc_sc=torch.zeros(2000,1,device=device)

def sc_loss():
    c=col_sc.clone().requires_grad_(True)
    pr,pi=m_sc(c)
    dpr=torch.autograd.grad(pr.sum(),c,create_graph=True)[0]
    dpi=torch.autograd.grad(pi.sum(),c,create_graph=True)[0]
    pr_t,pr_x=dpr[:,1:2],dpr[:,0:1]
    pi_t,pi_x=dpi[:,1:2],dpi[:,0:1]
    pr_xx=torch.autograd.grad(pr_x.sum(),c,create_graph=True)[0][:,0:1]
    pi_xx=torch.autograd.grad(pi_x.sum(),c,create_graph=True)[0][:,0:1]
    # iψ_t = -(1/2)ψ_xx → real: -pi_t = -(1/2)pr_xx → pi_t - (1/2)pr_xx=0
    #                       imag:  pr_t = -(1/2)pi_xx → pr_t + (1/2)pi_xx=0... wait
    # TDSE split: -ψ_i_t - (1/2)ψ_r_xx = 0, ψ_r_t - (1/2)ψ_i_xx = 0
    res_r=(-pi_t - 0.5*pr_xx).pow(2).mean()
    res_i=(pr_t - 0.5*pi_xx).pow(2).mean()
    pr_pred_ic,pi_pred_ic=m_sc(c_ic_sc)
    l_ic=F.mse_loss(pr_pred_ic,pr_ic)+F.mse_loss(pi_pred_ic,pi_ic)
    pr_bcl,pi_bcl=m_sc(c_bcl_sc); pr_bcr,pi_bcr=m_sc(c_bcr_sc)
    l_bc=F.mse_loss(pr_bcl,u_bc_sc)+F.mse_loss(pi_bcl,u_bc_sc)+F.mse_loss(pr_bcr,u_bc_sc)+F.mse_loss(pi_bcr,u_bc_sc)
    return res_r+res_i+100*(l_ic+l_bc)

t0=time.time()
adam_train(m_sc, sc_loss, 8000, 5e-4)
lbfgs_refine(m_sc, sc_loss, 300)
elapsed_sc=time.time()-t0

xe=torch.linspace(0,1,200,device=device); te=torch.linspace(0,1,200,device=device)
X_sc,T_sc=torch.meshgrid(xe,te,indexing='ij')
ce_sc=torch.stack([X_sc.flatten(),T_sc.flatten()],1)
with torch.no_grad(): pr_p,pi_p=m_sc(ce_sc)
pr_p=pr_p.reshape(200,200).cpu(); pi_p=pi_p.reshape(200,200).cpu()
pr_true=torch.sin(math.pi*X_sc)*torch.cos(E1*T_sc)
pi_true=-torch.sin(math.pi*X_sc)*torch.sin(E1*T_sc)
pr_true=pr_true.cpu(); pi_true=pi_true.cpu()
l2_r=rel_l2(pr_p,pr_true); l2_i=rel_l2(pi_p,pi_true)
# Norm conservation: integral of |psi|^2 over x at each t ≈ 0.5
norm_vals=(pr_p**2+pi_p**2).mean(dim=0)  # mean over x
norm_drift=(norm_vals-norm_vals[0]).abs().max().item()
status_sc='PASS' if l2_r<0.1 and l2_i<0.1 else 'FAIL'
print(f'L2_real={l2_r:.4e} L2_imag={l2_i:.4e} norm_drift={norm_drift:.4e} {status_sc} {elapsed_sc:.0f}s')
results['b6_schrodinger']={'l2_real':l2_r,'l2_imag':l2_i,'norm_drift':norm_drift,'time':elapsed_sc,'status':status_sc}

fig,ax=plt.subplots(2,3,figsize=(14,6))
ax[0,0].contourf(T_sc.cpu(),X_sc.cpu(),pr_true,50,cmap='RdBu_r'); ax[0,0].set_title('ψ_real analytic')
ax[0,1].contourf(T_sc.cpu(),X_sc.cpu(),pr_p,50,cmap='RdBu_r'); ax[0,1].set_title('ψ_real rippl')
ax[0,2].contourf(T_sc.cpu(),X_sc.cpu(),(pr_p-pr_true).abs(),50,cmap='hot_r'); ax[0,2].set_title(f'Error L2={l2_r:.2e}')
ax[1,0].contourf(T_sc.cpu(),X_sc.cpu(),pi_true,50,cmap='RdBu_r'); ax[1,0].set_title('ψ_imag analytic')
ax[1,1].contourf(T_sc.cpu(),X_sc.cpu(),pi_p,50,cmap='RdBu_r'); ax[1,1].set_title('ψ_imag rippl')
ax[1,2].contourf(T_sc.cpu(),X_sc.cpu(),(pi_p-pi_true).abs(),50,cmap='hot_r'); ax[1,2].set_title(f'Error L2={l2_i:.2e}')
plt.tight_layout(); plt.savefig('b6_schrodinger.png',dpi=100); plt.show()

## B7 — Eikonal Equation 2D

In [ ]:
# CELL 10 — Eikonal: |∇V|² = 1, source at (0.5,0.5)
# Analytic: V(x,y) = sqrt((x-0.5)²+(y-0.5)²)
print('=== B7: Eikonal 2D ===')
m_eik=MLP(2,1,64,5).to(device)
col_eik=sobol(10000,2)
# BC: V=0 at source point approximately — enforce at ring around source
theta=torch.linspace(0,2*math.pi,500,device=device)
r_src=0.02
x_src=0.5+r_src*torch.cos(theta); y_src=0.5+r_src*torch.sin(theta)
c_src=torch.stack([x_src,y_src],1)
v_src=torch.full((500,1),r_src,device=device)

def eik_loss():
    c=col_eik.clone().requires_grad_(True)
    V=m_eik(c)
    dV=torch.autograd.grad(V.sum(),c,create_graph=True)[0]
    Vx,Vy=dV[:,0:1],dV[:,1:2]
    res=(Vx**2+Vy**2-1).pow(2).mean()
    l_src=F.mse_loss(m_eik(c_src),v_src)
    # Causality: V should be positive
    l_pos=F.relu(-V).pow(2).mean()
    return res+1000*l_src+10*l_pos

t0=time.time()
adam_train(m_eik, eik_loss, 8000, 5e-4)
lbfgs_refine(m_eik, eik_loss, 300)
elapsed_eik=time.time()-t0

xe=torch.linspace(0,1,100,device=device); ye=torch.linspace(0,1,100,device=device)
X_eik,Y_eik=torch.meshgrid(xe,ye,indexing='ij')
ce_eik=torch.stack([X_eik.flatten(),Y_eik.flatten()],1)
with torch.no_grad(): V_pred=m_eik(ce_eik).reshape(100,100).cpu()
V_true=torch.sqrt((X_eik-0.5)**2+(Y_eik-0.5)**2).cpu()
# Exclude source region
mask=(V_true>0.05)
l2_eik=rel_l2(V_pred[mask],V_true[mask])
status_eik='PASS' if l2_eik<0.1 else 'FAIL'
print(f'L2={l2_eik:.4e} {status_eik} {elapsed_eik:.0f}s')
results['b7_eikonal2d']={'l2':l2_eik,'time':elapsed_eik,'status':status_eik}

fig,ax=plt.subplots(1,3,figsize=(14,4))
ax[0].contourf(X_eik.cpu(),Y_eik.cpu(),V_true,50,cmap='viridis'); ax[0].set_title('Analytic')
ax[1].contourf(X_eik.cpu(),Y_eik.cpu(),V_pred,50,cmap='viridis'); ax[1].set_title('rippl')
ax[2].contourf(X_eik.cpu(),Y_eik.cpu(),(V_pred-V_true).abs(),50,cmap='hot_r'); ax[2].set_title(f'Error L2={l2_eik:.2e}')
plt.tight_layout(); plt.savefig('b7_eikonal2d.png',dpi=100); plt.show()

## B8 — Digital Twin Parameter Recovery

In [ ]:
# CELL 11 — Digital Twin: recover alpha from heat equation sensor data
# True alpha=0.1, start guess=0.5, recover via joint optimization
print('=== B8: Digital Twin Parameter Recovery ===')
alpha_true=0.1
alpha_param=torch.nn.Parameter(torch.tensor(0.5))
m_dt=MLP(2,1,50,4).to(device)

# Generate synthetic sensor data from analytic solution
x_sens=torch.rand(200,1,device=device)
t_sens=torch.rand(200,1,device=device)
c_sens=torch.cat([x_sens,t_sens],1)
with torch.no_grad():
    u_sens=torch.sin(math.pi*x_sens)*torch.exp(-alpha_true*math.pi**2*t_sens)
u_sens=u_sens+0.001*torch.randn_like(u_sens)  # small noise

col_dt=sobol(5000,2)
x_ic_dt=torch.rand(1000,1,device=device)
c_ic_dt=torch.cat([x_ic_dt,torch.zeros(1000,1,device=device)],1)
u_ic_dt=torch.sin(math.pi*x_ic_dt)
t_bc_dt=torch.rand(1000,1,device=device)
c_bcl_dt=torch.cat([torch.zeros(1000,1,device=device),t_bc_dt],1)
c_bcr_dt=torch.cat([torch.ones(1000,1,device=device),t_bc_dt],1)
u_bc_dt=torch.zeros(1000,1,device=device)

opt_dt=torch.optim.Adam([{'params':m_dt.parameters(),'lr':5e-4},{'params':[alpha_param],'lr':1e-2}])
sched_dt=torch.optim.lr_scheduler.ReduceLROnPlateau(opt_dt,patience=200,factor=0.5,min_lr=1e-6)
alpha_history=[]
t0=time.time()
print('Joint training...')
for ep in range(8000):
    opt_dt.zero_grad()
    c=col_dt.clone().requires_grad_(True)
    u=m_dt(c)
    du=torch.autograd.grad(u.sum(),c,create_graph=True)[0]
    u_t,u_x=du[:,1:2],du[:,0:1]
    u_xx=torch.autograd.grad(u_x.sum(),c,create_graph=True)[0][:,0:1]
    alpha_pos=F.softplus(alpha_param)  # ensure positive
    res=(u_t-alpha_pos*u_xx).pow(2).mean()
    l_ic=F.mse_loss(m_dt(c_ic_dt),u_ic_dt)
    l_bc=F.mse_loss(m_dt(c_bcl_dt),u_bc_dt)+F.mse_loss(m_dt(c_bcr_dt),u_bc_dt)
    l_data=F.mse_loss(m_dt(c_sens),u_sens)
    loss=res+100*(l_ic+l_bc)+10*l_data
    loss.backward()
    torch.nn.utils.clip_grad_norm_(m_dt.parameters(),1.0)
    opt_dt.step(); sched_dt.step(loss)
    alpha_history.append(F.softplus(alpha_param).item())
    if ep%2000==0:
        print(f'  ep{ep}: loss={loss.item():.3e} alpha={F.softplus(alpha_param).item():.4f} (true={alpha_true})')

elapsed_dt=time.time()-t0
alpha_identified=F.softplus(alpha_param).item()
alpha_error=abs(alpha_identified-alpha_true)/alpha_true
status_dt='PASS' if alpha_error<0.05 else 'FAIL'  # within 5%
print(f'Identified alpha={alpha_identified:.4f} true={alpha_true} error={alpha_error*100:.2f}% {status_dt} {elapsed_dt:.0f}s')
results['b8_digital_twin']={'identified_alpha':alpha_identified,'true_alpha':alpha_true,'relative_error':alpha_error,'time':elapsed_dt,'status':status_dt}

plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.plot(alpha_history); plt.axhline(alpha_true,c='r',ls='--',label=f'True={alpha_true}'); plt.title('Alpha convergence'); plt.legend()
plt.subplot(1,2,2)
xe2=torch.linspace(0,1,100,device=device); te2=torch.linspace(0,1,100,device=device)
X2,T2=torch.meshgrid(xe2,te2,indexing='ij')
ce2=torch.stack([X2.flatten(),T2.flatten()],1)
with torch.no_grad(): up2=m_dt(ce2).reshape(100,100).cpu()
ut2=(torch.sin(math.pi*X2)*torch.exp(-alpha_true*math.pi**2*T2)).cpu()
plt.contourf(T2.cpu(),X2.cpu(),(up2-ut2).abs(),50,cmap='hot_r'); plt.colorbar(); plt.title('Field error')
plt.tight_layout(); plt.savefig('b8_digital_twin.png',dpi=100); plt.show()

## B9 — Uncertainty Quantification (MC Dropout)

In [ ]:
# CELL 12 — UQ: MC Dropout on heat equation
# Verify: confidence intervals contain analytic solution
print('=== B9: UQ MC Dropout ===')

class MLPDropout(nn.Module):
    def __init__(self, p=0.1):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(2,50),nn.Tanh(),nn.Dropout(p),
            nn.Linear(50,50),nn.Tanh(),nn.Dropout(p),
            nn.Linear(50,50),nn.Tanh(),nn.Dropout(p),
            nn.Linear(50,50),nn.Tanh(),nn.Dropout(p),
            nn.Linear(50,1)
        )
        for m in self.modules():
            if isinstance(m,nn.Linear): nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self,x): return self.net(x)

m_uq=MLPDropout(p=0.05).to(device)
col_uq=sobol(8000,2)
x_ic_uq=torch.rand(2000,1,device=device)
c_ic_uq=torch.cat([x_ic_uq,torch.zeros(2000,1,device=device)],1)
u_ic_uq=torch.sin(math.pi*x_ic_uq)
t_bc_uq=torch.rand(2000,1,device=device)
c_bcl_uq=torch.cat([torch.zeros(2000,1,device=device),t_bc_uq],1)
c_bcr_uq=torch.cat([torch.ones(2000,1,device=device),t_bc_uq],1)
u_bc_uq=torch.zeros(2000,1,device=device)

def uq_loss():
    c=col_uq.clone().requires_grad_(True)
    u=m_uq(c)
    du=torch.autograd.grad(u.sum(),c,create_graph=True)[0]
    u_t,u_x=du[:,1:2],du[:,0:1]
    u_xx=torch.autograd.grad(u_x.sum(),c,create_graph=True)[0][:,0:1]
    res=(u_t-0.1*u_xx).pow(2).mean()
    l_ic=F.mse_loss(m_uq(c_ic_uq),u_ic_uq)
    l_bc=F.mse_loss(m_uq(c_bcl_uq),u_bc_uq)+F.mse_loss(m_uq(c_bcr_uq),u_bc_uq)
    return res+100*(l_ic+l_bc)

t0=time.time()
adam_train(m_uq, uq_loss, 6000, 5e-4)
elapsed_uq=time.time()-t0

# MC Dropout inference: 50 forward passes
m_uq.train()  # keep dropout active
x_test=torch.linspace(0,1,100,device=device)
t_fixed=torch.full((100,),0.5,device=device)
c_test=torch.stack([x_test,t_fixed],1)
samples=[]
with torch.no_grad():
    for _ in range(50): samples.append(m_uq(c_test).cpu())
samples=torch.stack(samples,0)  # (50,100,1)
mean_pred=samples.mean(0).squeeze()
std_pred=samples.std(0).squeeze()
u_analytic=torch.sin(math.pi*x_test)*torch.exp(-0.1*math.pi**2*0.5)
u_analytic=u_analytic.cpu()

# Check: analytic within mean±2std
in_ci=((u_analytic>mean_pred-2*std_pred)&(u_analytic<mean_pred+2*std_pred)).float().mean().item()
mean_std=std_pred.mean().item()
status_uq='PASS' if in_ci>0.8 else 'FAIL'
print(f'CI coverage={in_ci:.2f} mean_std={mean_std:.4e} {status_uq} {elapsed_uq:.0f}s')
results['b9_uq']={'ci_coverage':in_ci,'mean_std':mean_std,'time':elapsed_uq,'status':status_uq}

plt.figure(figsize=(10,4))
x_np=x_test.cpu().numpy()
plt.plot(x_np,u_analytic.numpy(),'k-',lw=2,label='Analytic')
plt.plot(x_np,mean_pred.numpy(),'b-',lw=2,label='MC mean')
plt.fill_between(x_np,(mean_pred-2*std_pred).numpy(),(mean_pred+2*std_pred).numpy(),alpha=0.3,label='±2σ')
plt.title(f'UQ Heat Eq t=0.5 — CI coverage={in_ci:.0%}'); plt.legend()
plt.tight_layout(); plt.savefig('b9_uq.png',dpi=100); plt.show()

## B10 — CSG Domain (Annulus Laplace)

In [ ]:
# CELL 13 — Laplace on annulus: r_in=0.3, r_out=1.0
# u=0 on inner, u=1 on outer, analytic: u=(log(r)-log(r_in))/(log(r_out)-log(r_in))
print('=== B10: CSG Annulus Laplace ===')
r_in=0.3; r_out=1.0
m_csg=MLP(2,1,64,5).to(device)

# Sample interior of annulus via rejection
def sample_annulus(n):
    pts=[]
    while len(pts)<n:
        p=torch.rand(n*4,2)*2-1  # [-1,1]^2
        r=p.norm(dim=1)
        mask=(r>r_in)&(r<r_out)
        pts.append(p[mask])
    return torch.cat(pts,0)[:n].to(device)

def sample_circle_boundary(n, r):
    theta=torch.rand(n)*2*math.pi
    return torch.stack([r*torch.cos(theta),r*torch.sin(theta)],1).to(device)

col_csg=sample_annulus(8000)
c_inner=sample_circle_boundary(1000,r_in)
c_outer=sample_circle_boundary(1000,r_out)
u_inner=torch.zeros(1000,1,device=device)
u_outer=torch.ones(1000,1,device=device)

def csg_loss():
    c=col_csg.clone().requires_grad_(True)
    u=m_csg(c)
    du=torch.autograd.grad(u.sum(),c,create_graph=True)[0]
    u_x,u_y=du[:,0:1],du[:,1:2]
    u_xx=torch.autograd.grad(u_x.sum(),c,create_graph=True)[0][:,0:1]
    u_yy=torch.autograd.grad(u_y.sum(),c,create_graph=True)[0][:,1:2]
    res=(u_xx+u_yy).pow(2).mean()
    l_bc=F.mse_loss(m_csg(c_inner),u_inner)+F.mse_loss(m_csg(c_outer),u_outer)
    return res+1000*l_bc

t0=time.time()
adam_train(m_csg, csg_loss, 8000, 5e-4)
lbfgs_refine(m_csg, csg_loss, 300)
elapsed_csg=time.time()-t0

# Eval on annulus grid
pts_eval=sample_annulus(3000)
r_eval=pts_eval.norm(dim=1,keepdim=True)
u_true_csg=(torch.log(r_eval)-math.log(r_in))/(math.log(r_out)-math.log(r_in))
with torch.no_grad(): u_pred_csg=m_csg(pts_eval)
l2_csg=rel_l2(u_pred_csg.cpu(),u_true_csg.cpu())
status_csg='PASS' if l2_csg<0.1 else 'FAIL'
print(f'L2={l2_csg:.4e} {status_csg} {elapsed_csg:.0f}s')
results['b10_csg_annulus']={'l2':l2_csg,'time':elapsed_csg,'status':status_csg}

fig,ax=plt.subplots(1,2,figsize=(10,4))
sc1=ax[0].scatter(pts_eval[:,0].cpu(),pts_eval[:,1].cpu(),c=u_true_csg.cpu(),s=2,cmap='viridis'); ax[0].set_title('Analytic'); plt.colorbar(sc1,ax=ax[0])
sc2=ax[1].scatter(pts_eval[:,0].cpu(),pts_eval[:,1].cpu(),c=u_pred_csg.cpu(),s=2,cmap='viridis'); ax[1].set_title(f'rippl L2={l2_csg:.2e}'); plt.colorbar(sc2,ax=ax[1])
plt.tight_layout(); plt.savefig('b10_csg_annulus.png',dpi=100); plt.show()

## B11 — Multi-Fidelity Sensor Fusion

In [ ]:
# CELL 14 — Multi-fidelity: sparse clean + dense noisy + PDE regularizer
print('=== B11: Multi-Fidelity Sensor Fusion ===')
alpha_mf=0.1
m_mf=MLP(2,1,50,4).to(device)

# High fidelity: 50 clean sensor points
x_hf=torch.rand(50,1,device=device); t_hf=torch.rand(50,1,device=device)
c_hf=torch.cat([x_hf,t_hf],1)
u_hf=torch.sin(math.pi*x_hf)*torch.exp(-alpha_mf*math.pi**2*t_hf)

# Low fidelity: 200 noisy sensor points
x_lf=torch.rand(200,1,device=device); t_lf=torch.rand(200,1,device=device)
c_lf=torch.cat([x_lf,t_lf],1)
u_lf=torch.sin(math.pi*x_lf)*torch.exp(-alpha_mf*math.pi**2*t_lf)+0.05*torch.randn(200,1,device=device)

# PDE collocation
col_mf=sobol(5000,2)
x_ic_mf=torch.rand(500,1,device=device)
c_ic_mf=torch.cat([x_ic_mf,torch.zeros(500,1,device=device)],1)
u_ic_mf=torch.sin(math.pi*x_ic_mf)

def mf_loss():
    c=col_mf.clone().requires_grad_(True)
    u=m_mf(c)
    du=torch.autograd.grad(u.sum(),c,create_graph=True)[0]
    u_t,u_x=du[:,1:2],du[:,0:1]
    u_xx=torch.autograd.grad(u_x.sum(),c,create_graph=True)[0][:,0:1]
    pde=(u_t-alpha_mf*u_xx).pow(2).mean()
    # Multi-fidelity: high weight on HF, lower on LF
    l_hf=10.0*F.mse_loss(m_mf(c_hf),u_hf)   # high fidelity weight
    l_lf=1.0*F.mse_loss(m_mf(c_lf),u_lf)    # low fidelity weight
    l_ic=F.mse_loss(m_mf(c_ic_mf),u_ic_mf)
    return pde+l_hf+l_lf+10*l_ic

t0=time.time()
adam_train(m_mf, mf_loss, 5000, 5e-4)
lbfgs_refine(m_mf, mf_loss, 200)
elapsed_mf=time.time()-t0

xe=torch.linspace(0,1,100,device=device); te=torch.linspace(0,1,100,device=device)
X_mf,T_mf=torch.meshgrid(xe,te,indexing='ij')
ce_mf=torch.stack([X_mf.flatten(),T_mf.flatten()],1)
with torch.no_grad(): up_mf=m_mf(ce_mf).reshape(100,100).cpu()
ut_mf=(torch.sin(math.pi*X_mf)*torch.exp(-alpha_mf*math.pi**2*T_mf)).cpu()
l2_mf=rel_l2(up_mf,ut_mf)
status_mf='PASS' if l2_mf<0.05 else 'FAIL'
print(f'L2={l2_mf:.4e} {status_mf} {elapsed_mf:.0f}s')
results['b11_multifidelity']={'l2':l2_mf,'time':elapsed_mf,'status':status_mf}

## B12 — DeepXDE Comparison

In [ ]:
# CELL 15 — DeepXDE heat 1D for direct comparison
print('=== B12: DeepXDE Comparison ===')
try:
    import deepxde as dde
    import numpy as np
    alpha_dde=0.1
    def pde(x,y):
        dy_t=dde.grad.jacobian(y,x,i=0,j=1)
        dy_xx=dde.grad.hessian(y,x,i=0,j=0)
        return dy_t-alpha_dde*dy_xx
    geom=dde.geometry.Interval(0,1)
    td=dde.geometry.TimeDomain(0,1)
    gt=dde.geometry.GeometryXTime(geom,td)
    bc=dde.icbc.DirichletBC(gt,lambda x:0,lambda x,ob:ob)
    ic=dde.icbc.IC(gt,lambda x:np.sin(np.pi*x[:,0:1]),lambda x,oi:oi)
    data=dde.data.TimePDE(gt,pde,[bc,ic],num_domain=8000,num_boundary=2000,num_initial=2000)
    net=dde.nn.FNN([2]+[50]*5+[1],'tanh','Glorot normal')
    mdl=dde.Model(data,net)
    t0=time.time()
    mdl.compile('adam',lr=5e-4); mdl.train(iterations=8000,display_every=4000)
    mdl.compile('L-BFGS'); mdl.train(display_every=500)
    elapsed_dde=time.time()-t0
    xe_np=np.linspace(0,1,200); te_np=np.linspace(0,1,200)
    X_np,T_np=np.meshgrid(xe_np,te_np,indexing='ij')
    xt=np.stack([X_np.flatten(),T_np.flatten()],1)
    u_dde=mdl.predict(xt).reshape(200,200)
    u_true_np=np.sin(np.pi*X_np)*np.exp(-alpha_dde*np.pi**2*T_np)
    l2_dde=np.sqrt(((u_dde-u_true_np)**2).sum())/np.sqrt((u_true_np**2).sum())
    print(f'DeepXDE L2={l2_dde:.4e} {elapsed_dde:.0f}s')
    results['b12_deepxde']={'l2':float(l2_dde),'time':elapsed_dde}
    # Comparison
    rippl_l2=results.get('b2_heat1d',{}).get('l2',None)
    if rippl_l2:
        print(f'rippl L2={rippl_l2:.4e} vs DeepXDE L2={l2_dde:.4e}')
        print(f'rippl faster: {results["b2_heat1d"]["time"]<elapsed_dde}')
        results['b12_deepxde']['rippl_l2']=rippl_l2
        results['b12_deepxde']['rippl_faster']=results['b2_heat1d']['time']<elapsed_dde
except Exception as e:
    print(f'DeepXDE benchmark failed: {e}')
    results['b12_deepxde']={'status':'SKIPPED','error':str(e)}

## Final Report

In [ ]:
# CELL 16 — Final report
print('='*60)
print('RIPPL FULL VALIDATION BENCHMARK RESULTS')
print('='*60)
benchmarks=[
    ('B1','Physics Sanity','b1_sanity','residual'),
    ('B2','Heat 1D','b2_heat1d','l2'),
    ('B3','Wave 1D Causal','b3_wave1d','l2'),
    ('B4','Burgers 1D','b4_burgers1d','mean_l2'),
    ('B5','Allen-Cahn 1D','b5_allen_cahn','ic_l2'),
    ('B6','Schrödinger','b6_schrodinger','l2_real'),
    ('B7','Eikonal 2D','b7_eikonal2d','l2'),
    ('B8','Digital Twin','b8_digital_twin','relative_error'),
    ('B9','UQ MC Dropout','b9_uq','ci_coverage'),
    ('B10','CSG Annulus','b10_csg_annulus','l2'),
    ('B11','Multi-Fidelity','b11_multifidelity','l2'),
]
passed=0; total=0
for code,name,key,metric in benchmarks:
    if key in results:
        r=results[key]
        val=r.get(metric,'N/A')
        status=r.get('status','N/A')
        t=r.get('time',0)
        print(f'{code} {name:20s}: {metric}={val:.4e}  {status}  {t:.0f}s')
        if status=='PASS': passed+=1
        total+=1
    else:
        print(f'{code} {name:20s}: NOT RUN')

if 'b12_deepxde' in results and 'l2' in results['b12_deepxde']:
    r=results['b12_deepxde']
    print(f'B12 DeepXDE comparison: L2={r["l2"]:.4e}  {r["time"]:.0f}s  rippl_faster={r.get("rippl_faster","N/A")}')

print('='*60)
print(f'{passed}/{total} benchmarks PASSED')
with open('benchmark_results.json','w') as f:
    json.dump(results,f,indent=2)
print('Saved benchmark_results.json')
print('Download this file and paste contents to project lead.')